In [1]:
import os
import re
import ftfy
import time
import unicodedata
import pandas as pd

# KONFIGURASI
INPUT_FILE = "../dataset/dataset.csv"
OUTPUT_FOLDER = "../output"
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "dataset_preprocessing.csv")

TEXT_COLUMN = "comment"


# MEMBUAT FOLDER OUTPUT
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# LOAD DATASET
df = pd.read_csv(INPUT_FILE)


# HOMOGLYPH
mapping = {

    "А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H",
    "О":"O","Р":"P","С":"C","Т":"T","Х":"X","У":"Y",

    "а":"a","е":"e","о":"o","р":"p","с":"c",
    "х":"x","у":"y","к":"k","м":"m","н":"h",

    "Α":"A","Β":"B","Ε":"E","Η":"H",
    "Ι":"I","Κ":"K","Μ":"M","Ν":"N",
    "Ο":"O","Ρ":"P","Τ":"T","Χ":"X"

}

TRANSLATE_TABLE = str.maketrans(mapping)


# COMPILE REGEX
HTML_RE = re.compile(r"<.*?>")
URL_RE = re.compile(r"http\S+")
WWW_RE = re.compile(r"www\S+")
MENTION_RE = re.compile(r"@\w+")
EMOJI_RE = re.compile(r"[\U00010000-\U0010ffff]", flags=re.UNICODE)
SPECIAL_RE = re.compile(r"[^A-Za-z0-9\s]")
SPACE_RE = re.compile(r"\s+")


# UNICODE FIX
def unicode_fix(text):

    text = ftfy.fix_text(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(TRANSLATE_TABLE)

    return text


# CLEANING
def cleaning(text):

    text = unicode_fix(text)

    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = WWW_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = EMOJI_RE.sub(" ", text)
    text = SPECIAL_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text)

    return text.strip()


# CASE FOLDING
def casefold(text):

    return text.lower()


# PREPROCESSING
start = time.time()

df["cleaning"] = (
    df[TEXT_COLUMN]
    .fillna("")
    .apply(cleaning)
)

df["casefolding"] = (
    df["cleaning"]
    .apply(casefold)
)

df["hasil_preprocessing"] = df["casefolding"]

elapsed = time.time() - start


# SAVE
df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# OUTPUT
print("="*60)
print("PREPROCESSING SELESAI")
print("="*60)
print(f"Jumlah Data        : {len(df)}")
print(f"Waktu Proses       : {elapsed:.2f} detik")
print(f"Hasil Disimpan     : {OUTPUT_FILE}")
print("="*60)

print(df[
    [
        TEXT_COLUMN,
        "cleaning",
        "casefolding",
        "hasil_preprocessing"
    ]
].head())

PREPROCESSING SELESAI
Jumlah Data        : 20807
Waktu Proses       : 0.78 detik
Hasil Disimpan     : ../output\dataset_preprocessing.csv
                                             comment  \
0      AЕR𝐎𝟴𝟴 gua main dikit aja, saldo lngsg nambah   
1  Sebelumnya bermain di tempat lain terasa kuran...   
2   AЕR𝐎𝟴𝟴 aku sih gmpang bgt dapet WEDEY gede dsini   
3    AЕR𝐎𝟴𝟴 gua cba bentar, eh lgnsung dapet jackpot   
4            AЕR𝐎𝟴𝟴 aku cba maen, malah jackpot gede   

                                            cleaning  \
0       AERO88 gua main dikit aja saldo lngsg nambah   
1  Sebelumnya bermain di tempat lain terasa kuran...   
2   AERO88 aku sih gmpang bgt dapet WEDEY gede dsini   
3     AERO88 gua cba bentar eh lgnsung dapet jackpot   
4             AERO88 aku cba maen malah jackpot gede   

                                         casefolding  \
0       aero88 gua main dikit aja saldo lngsg nambah   
1  sebelumnya bermain di tempat lain terasa kuran...   
2   aero88 aku sih g